# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIRˆ2 dataset using the `mlcroissant` library and the Croissant schema.

### Dataset Source
The Croissant schema defining the dataset can be found at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the dataset metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List all record sets in the dataset by their `@id`, and show their fields and columns. This allows selection of which parts to extract for further analysis.

In [ ]:
# List the record sets available in the dataset and their @id
print("Available record sets:")
record_set_ids = []
for rs in metadata.record_sets:
    print(f"- @id: {rs.id}, name: {getattr(rs, 'name', '')}")
    record_set_ids.append(rs.id)

# For each record set, list its fields (@id and name)
record_set_fields = {}
for rs in metadata.record_sets:
    print(f"\nRecordSet @id: {rs.id}")
    fields = []
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"  - Field @id: {field.id}, name: {getattr(field, 'name', '')}, dataType: {getattr(field, 'data_type', '')}")
            fields.append(field.id)
    else:
        print("  (No fields declared)")
    record_set_fields[rs.id] = fields

# Save for later use
if record_set_ids:
    main_rs_id = record_set_ids[0]
    example_fields = record_set_fields[main_rs_id]
else:
    main_rs_id = None
    example_fields = []

## 3. Data Extraction
Load data from each record set of interest into a pandas DataFrame for analysis. All entity references use their `@id`.

> You can change the value of `record_set_ids` below to only load specific record sets, if desired.

In [ ]:
# Extract data from each record set by @id, using mlcroissant
dataframes = {}
# Use the list of record set @ids identified above
for record_set_id in record_set_ids:
    print(f"\nLoading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records with columns: {df.columns.tolist()}")

if main_rs_id and main_rs_id in dataframes:
    print("\nSample rows from record set:")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply exploratory steps, such as filtering, normalizing, and grouping with only `@id` references. Modify IDs below as appropriate for your dataset.

The code assumes a numeric field exists (e.g., age, diagnosis interval, etc.), and a group/category field (e.g., anatomical location, MSI status). Adjust the code as suggested in the comments based on the field IDs and content you see.

In [ ]:
# Example EDA using the first available record set and likely field IDs.
# Modify `numeric_field_id` and `group_field_id` based on your inspection above (see printed IDs and names).

# Use main_rs_id and available fields for an example
import numpy as np

record_set_id = main_rs_id
df = dataframes.get(record_set_id)

# EXAMPLE: Suppose we have an interval column (e.g., time between diagnoses) and anatomical group, identified by their field @ids
if df is not None and not df.empty:
    # Pick a numeric field @id (update this if needed based on your fields)
    # Replace with your field e.g., 'interval_years' if present
    
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        if any(x in col.lower() for x in ['age', 'interval', 'year', 'duration', 'count']):
            numeric_field_id = col
        elif any(x in col.lower() for x in ['site', 'anatomical', 'location', 'msi', 'gender', 'sex', 'status']):
            group_field_id = col

    print(f"Selected numeric field @id: {numeric_field_id}")
    print(f"Selected group field @id: {group_field_id}")
    
    if numeric_field_id is not None and np.issubdtype(df[numeric_field_id].dtype, np.number):
        threshold = df[numeric_field_id].quantile(0.25)  # e.g. lower quartile
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical field if present
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'count']).reset_index()
            print(f"\nGrouped data by {group_field_id}:")
            display(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA. Please inspect the DataFrame columns above and update the field @id in this cell.")
else:
    print("No data loaded for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields. Use the field and record set `@id`s.

Below is a template for common visualizations. Update the IDs using your field inspection above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Use the same selected fields as above
if df is not None and not df.empty and numeric_field_id is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIRˆ2 Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer dataset via its Croissant schema, enumerated its entities using their `@id`, and performed EDA and basic visualization. You can further analyze and process the data by referencing entity `@id`s and following this workflow.